In [7]:
import xarray as xr
ds = xr.open_zarr('gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr',consolidated=True,)

In [8]:
import xarray as xr
from dask.diagnostics import ProgressBar  # Add this import

# 2. Define variables to keep
vars_to_keep = [
    'total_precipitation_6hr',
    '2m_temperature', '2m_dewpoint_temperature',
    'surface_pressure', 'mean_sea_level_pressure',
    '10m_u_component_of_wind', '10m_v_component_of_wind',
    'u_component_of_wind', 'v_component_of_wind',
    'specific_humidity', 'relative_humidity',
    'total_column_water_vapour', 'total_cloud_cover',
    'mean_surface_net_short_wave_radiation_flux',
    'mean_surface_latent_heat_flux',
    'vertical_velocity', 'potential_vorticity',
    'boundary_layer_height',
    'geopotential_at_surface', 'land_sea_mask'
]

ds_subset = ds[vars_to_keep].sel(time=slice("1959", "2023")).where( 
    (ds.longitude >= 335) | (ds.longitude <= 50), 
    drop=True 
).sel(latitude=slice(75, 30)) 


In [ ]:
import xarray as xr
import numpy as np

class WeatherBench2DataLoader:
    def __init__(self, zarr_path, vars_to_keep, batch_size=32):
        self.ds = xr.open_zarr(
            zarr_path,
            consolidated=True,
            storage_options={"token": "cloud"},
            chunks={'time': batch_size}  # Chunk by batch size
        )
        
        # Apply spatial and variable filtering
        self.ds = self.ds[vars_to_keep].where(
            (self.ds.longitude >= 335) | (self.ds.longitude <= 50),
            drop=True
        ).sel(latitude=slice(75, 30))
        
    def get_batch(self, time_indices):
        """Get a batch of data for specific time indices"""
        return self.ds.isel(time=time_indices).load()
    
    def __len__(self):
        return len(self.ds.time)

# Usage in training
data_loader = WeatherBench2DataLoader(
    "gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr",
    vars_to_keep,
    batch_size=32
)

# Get a batch (this loads only what you need)
batch_indices = np.arange(0, 32)  # First 32 time steps
batch_data = data_loader.get_batch(batch_indices)